# 📖 Notebook 4: Compaction Strategies

Cassandra's storage engine is fundamentally different from traditional databases. Instead of
updating data in place (like PostgreSQL's B-tree), Cassandra writes everything as **append-only**
operations using a **Log-Structured Merge Tree (LSM tree)**.

This makes writes extremely fast, but it means data piles up on disk. **Compaction** is the
background process that cleans up this mess.

## Learning Objectives

By the end of this notebook, you'll understand:
- How Cassandra's write path works (commit log → memtable → SSTable)
- What SSTables are and why they're immutable
- What tombstones are and why deletes don't really delete
- How compaction merges SSTables
- The three main compaction strategies and when to use each

## 🛠️ Setup

Make sure the cluster is running:

```bash
cd deep-dives/cassandra
docker-compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
from cassandra.cluster import Cluster
from cassandra.query import SimpleStatement
from tabulate import tabulate
import subprocess
import time

cluster = Cluster(["localhost"], port=9042)
session = cluster.connect()

session.execute("""
    CREATE KEYSPACE IF NOT EXISTS demo
    WITH REPLICATION = { 'class': 'SimpleStrategy', 'replication_factor': 3 }
""")
session.set_keyspace('demo')

print(f"Connected to: {cluster.metadata.cluster_name}")

## Step 1: The Write Path — How Data Gets to Disk

When you write data to Cassandra, it goes through three stages:

```
Client Write
    │
    ├──→ 1. Commit Log (on disk)     ← Write-ahead log for durability
    │                                   If the node crashes, data can be recovered
    │
    └──→ 2. Memtable (in memory)     ← Sorted in-memory buffer
              │                         Fast writes because it's just memory
              │
              └──→ 3. SSTable (on disk)  ← When memtable is full, flush to disk
                                          Immutable — never modified after creation
```

Key insight: **Every write is an append operation.** Updates don't modify existing data —
they create new entries. Deletes write a special marker called a **tombstone**.

Let's see this in action.

In [ ]:
# Create a table to demonstrate the write path
session.execute("""
    CREATE TABLE IF NOT EXISTS write_path_demo (
        id text PRIMARY KEY,
        value text,
        version int
    )
""")

# Insert a row
session.execute("INSERT INTO write_path_demo (id, value, version) VALUES ('key1', 'first', 1)")
print("Step 1: INSERT → commit log + memtable")

# Update the same row (this creates a NEW entry, doesn't modify the old one)
session.execute("INSERT INTO write_path_demo (id, value, version) VALUES ('key1', 'second', 2)")
print("Step 2: UPDATE → NEW entry in commit log + memtable (old entry still exists!)")

# Update again
session.execute("INSERT INTO write_path_demo (id, value, version) VALUES ('key1', 'third', 3)")
print("Step 3: UPDATE → ANOTHER new entry")

# Read — Cassandra figures out the latest version
row = session.execute("SELECT * FROM write_path_demo WHERE id = 'key1'").one()
print(f"\nRead result: value='{row.value}', version={row.version}")
print("\nCassandra returns 'third' because it has the latest timestamp.")
print("But internally, ALL THREE versions exist until compaction cleans them up!")

## Step 2: Tombstones — Why Deletes Don't Really Delete

Because SSTables are **immutable** (never modified after creation), Cassandra can't just
remove a row from an SSTable. Instead, it writes a **tombstone** — a special marker that says
"this row has been deleted."

The tombstone is kept for a period called `gc_grace_seconds` (default: 10 days). This gives
all replicas time to learn about the delete. After this period, compaction removes the
tombstone and the original data.

**Why this matters**: If you delete a lot of data, those tombstones pile up and slow down
reads until compaction cleans them up.

In [ ]:
# Demonstrate tombstones
session.execute("""
    CREATE TABLE IF NOT EXISTS tombstone_demo (
        partition_id text,
        row_id int,
        data text,
        PRIMARY KEY (partition_id, row_id)
    )
""")

# Insert 10 rows
for i in range(10):
    session.execute(f"INSERT INTO tombstone_demo (partition_id, row_id, data) VALUES ('p1', {i}, 'data_{i}')")
print("Inserted 10 rows")

# Delete 5 of them
for i in range(5):
    session.execute(f"DELETE FROM tombstone_demo WHERE partition_id = 'p1' AND row_id = {i}")
print("Deleted 5 rows")

# Read remaining rows
rows = session.execute("SELECT row_id, data FROM tombstone_demo WHERE partition_id = 'p1'")
remaining = list(rows)
print(f"\nVisible rows: {len(remaining)}")
for r in remaining:
    print(f"  row_id={r.row_id}, data={r.data}")

print("\n⚠️ Internally, the deleted rows still exist as tombstones!")
print("Cassandra must read through them to find the live rows.")
print(f"Tombstones are kept for gc_grace_seconds (default: 864000 = 10 days).")

> ⚠️ **Tombstone anti-pattern.** Using Cassandra as a queue (insert → read → delete) quickly fills partitions with tombstones that every read must scan past. By default Cassandra logs a warning at 1,000 tombstones per query and fails the query at 100,000. If you need queue-like behaviour, prefer TTL + `TimeWindowCompactionStrategy` (covered in Notebook 5).

## Step 3: Flushing Memtables to SSTables

When a memtable reaches a size threshold, Cassandra flushes it to disk as an **SSTable**
(Sorted String Table). Let's force a flush and inspect the result.

In [ ]:
# Insert data then force a flush to create SSTables
session.execute("""
    CREATE TABLE IF NOT EXISTS compaction_demo (
        id text PRIMARY KEY,
        value text
    ) WITH compaction = {'class': 'SizeTieredCompactionStrategy'}
""")

# Insert batch 1
for i in range(100):
    session.execute(f"INSERT INTO compaction_demo (id, value) VALUES ('key_{i}', 'batch1_value_{i}')")
print("Inserted batch 1 (100 rows)")

# Force flush to create an SSTable
result = subprocess.run(
    ["docker", "exec", "cassandra-node1", "nodetool", "flush", "demo", "compaction_demo"],
    capture_output=True, text=True
)
print("Flushed memtable → SSTable #1")

# Insert batch 2 (some updates to existing keys, some new keys)
for i in range(50, 150):
    session.execute(f"INSERT INTO compaction_demo (id, value) VALUES ('key_{i}', 'batch2_value_{i}')")
print("Inserted batch 2 (100 rows, 50 overlapping with batch 1)")

# Force flush again
result = subprocess.run(
    ["docker", "exec", "cassandra-node1", "nodetool", "flush", "demo", "compaction_demo"],
    capture_output=True, text=True
)
print("Flushed memtable → SSTable #2")

print("\nNow we have 2 SSTables on disk:")
print("  SSTable #1: keys 0-99 with batch1 values")
print("  SSTable #2: keys 50-149 with batch2 values")
print("  Keys 50-99 exist in BOTH SSTables (different versions)")

In [ ]:
# Check SSTable stats using nodetool
result = subprocess.run(
    ["docker", "exec", "cassandra-node1", "nodetool", "tablestats", "demo.compaction_demo"],
    capture_output=True, text=True
)

# Parse and display key stats
lines = result.stdout.strip().split('\n')
interesting_keys = ['SSTable count', 'Space used', 'Number of partitions', 
                    'Compacted partition', 'Pending compactions']

print("SSTable Stats for compaction_demo:")
print("=" * 50)
for line in lines:
    for key in interesting_keys:
        if key.lower() in line.lower():
            print(f"  {line.strip()}")
            break

## Step 4: Compaction — Cleaning Up SSTables

Over time, SSTables accumulate:
- **Multiple versions** of the same row (from updates)
- **Tombstones** (from deletes)
- **Redundant data** spread across many files

**Compaction** merges SSTables together, keeping only the latest version of each row
and removing expired tombstones. The result is fewer, cleaner SSTables.

```
Before compaction:           After compaction:
┌──────────────┐
│ SSTable #1   │             ┌──────────────┐
│ key1=v1      │             │ Merged Table  │
│ key2=v1      │  ──merge──→ │ key1=v2 (latest) │
│ key3=v1      │             │ key2=v1       │
└──────────────┘             │ key3=v1       │
┌──────────────┐             │ key4=v1       │
│ SSTable #2   │             └──────────────┘
│ key1=v2      │
│ key4=v1      │
└──────────────┘
```

In [ ]:
# Force compaction and observe the result
print("Forcing compaction on compaction_demo...")
result = subprocess.run(
    ["docker", "exec", "cassandra-node1", "nodetool", "compact", "demo", "compaction_demo"],
    capture_output=True, text=True
)
print("Compaction complete!")

# Check stats after compaction
result = subprocess.run(
    ["docker", "exec", "cassandra-node1", "nodetool", "tablestats", "demo.compaction_demo"],
    capture_output=True, text=True
)

print("\nSSTable Stats after compaction:")
print("=" * 50)
for line in result.stdout.strip().split('\n'):
    for key in interesting_keys:
        if key.lower() in line.lower():
            print(f"  {line.strip()}")
            break

print("\n✅ Multiple SSTables merged into fewer files.")
print("Duplicate key versions resolved — only latest version kept.")

In [ ]:
# Verify the data is correct after compaction — batch2 values should win for overlapping keys
rows = session.execute("SELECT id, value FROM compaction_demo WHERE id IN ('key_49', 'key_50', 'key_99', 'key_100')")

print("Data after compaction:")
table_data = [[r.id, r.value] for r in rows]
print(tabulate(table_data, headers=["key", "value"], tablefmt="grid"))
print("\nkey_49 → batch1 (only existed in SSTable #1)")
print("key_50 → batch2 (existed in both, batch2 is newer)")
print("key_99 → batch2 (existed in both, batch2 is newer)")
print("key_100 → batch2 (only existed in SSTable #2)")

## Step 5: Compaction Strategies

Cassandra offers different strategies for WHEN and HOW to compact SSTables.
The choice depends on your workload:

### 1. Size-Tiered Compaction (STCS) — Default
- **How**: Groups SSTables of similar size and merges them
- **Good for**: Write-heavy workloads
- **Bad for**: Read-heavy workloads (many SSTables to check)
- **Space**: Needs ~2× table size for temporary compaction space

### 2. Leveled Compaction (LCS)
- **How**: Organizes SSTables into levels (L0, L1, L2...). Each level is 10× bigger. SSTables in each level have non-overlapping key ranges.
- **Good for**: Read-heavy workloads (fewer SSTables to check per read)
- **Bad for**: Write-heavy workloads (more I/O from frequent compaction)
- **Space**: Only ~10% temporary overhead

### 3. Time-Window Compaction (TWCS)
- **How**: Groups SSTables by time window, only compacts within the same window
- **Good for**: Time-series data with TTL (data expires after a set time)
- **Bad for**: Data that gets updated frequently

In [ ]:
# Create tables with different compaction strategies

# STCS — good for write-heavy (e.g., logging, analytics events)
session.execute("""
    CREATE TABLE IF NOT EXISTS logs_stcs (
        day text,
        log_id timeuuid,
        message text,
        PRIMARY KEY (day, log_id)
    ) WITH compaction = {
        'class': 'SizeTieredCompactionStrategy',
        'min_threshold': 4,
        'max_threshold': 32
    }
""")
print("✅ logs_stcs: SizeTieredCompactionStrategy")
print("   → Best for: write-heavy workloads (logging, event streaming)")
print("   → Merges SSTables of similar size together")
print()

# LCS — good for read-heavy (e.g., user profiles, product catalog)
session.execute("""
    CREATE TABLE IF NOT EXISTS profiles_lcs (
        user_id bigint PRIMARY KEY,
        name text,
        email text
    ) WITH compaction = {
        'class': 'LeveledCompactionStrategy',
        'sstable_size_in_mb': 160
    }
""")
print("✅ profiles_lcs: LeveledCompactionStrategy")
print("   → Best for: read-heavy workloads (user profiles, lookups)")
print("   → Organizes SSTables into levels with non-overlapping ranges")
print("   → Reads check fewer SSTables (usually just 1!)")
print()

# TWCS — good for time-series with TTL (e.g., metrics, sensor data)
session.execute("""
    CREATE TABLE IF NOT EXISTS metrics_twcs (
        sensor_id text,
        ts timestamp,
        value double,
        PRIMARY KEY (sensor_id, ts)
    ) WITH compaction = {
        'class': 'TimeWindowCompactionStrategy',
        'compaction_window_unit': 'DAYS',
        'compaction_window_size': 1
    }
    AND default_time_to_live = 604800
""")
print("✅ metrics_twcs: TimeWindowCompactionStrategy")
print("   → Best for: time-series data that expires (metrics, IoT)")
print("   → Groups SSTables by time window (1 day here)")
print("   → When a window's TTL expires, the whole SSTable is dropped — very efficient!")
print("   → TTL = 604800 seconds (7 days)")

In [ ]:
# Insert data into each table and compare behavior
from cassandra.util import uuid_from_time
from datetime import datetime, timedelta

# Write-heavy table: insert many log entries
print("Writing 500 log entries (STCS table)...")
insert_log = session.prepare("INSERT INTO logs_stcs (day, log_id, message) VALUES (?, ?, ?)")
for i in range(500):
    ts = datetime(2024, 6, 15) + timedelta(seconds=i)
    session.execute(insert_log, ('2024-06-15', uuid_from_time(ts), f'Log entry {i}'))
print("Done!")

# Read-heavy table: insert profiles then read them many times
print("\nWriting 100 user profiles (LCS table)...")
insert_profile = session.prepare("INSERT INTO profiles_lcs (user_id, name, email) VALUES (?, ?, ?)")
for i in range(100):
    session.execute(insert_profile, (i, f'User {i}', f'user{i}@example.com'))
print("Done!")

# Time-series table: insert sensor readings with implicit TTL
print("\nWriting 200 sensor readings (TWCS table, TTL=7 days)...")
insert_metric = session.prepare("INSERT INTO metrics_twcs (sensor_id, ts, value) VALUES (?, ?, ?)")
for i in range(200):
    ts = datetime.now() - timedelta(hours=i)
    session.execute(insert_metric, ('sensor-1', ts, 20.0 + (i * 0.1)))
print("Done!")

print("\nAll three tables populated. Each uses a different compaction strategy")
print("optimized for its specific workload pattern.")

## Step 6: Compaction Strategy Decision Guide

Here's a quick reference for system design interviews:

```
                        ┌─────────────────────┐
                        │ What's your workload?│
                        └─────────┬───────────┘
                                  │
                    ┌─────────────┼─────────────┐
                    │             │             │
              Write-heavy    Read-heavy    Time-series
              (logs, events) (profiles)    (with TTL)
                    │             │             │
                    ▼             ▼             ▼
                  STCS          LCS          TWCS
```

| Strategy | Write Speed | Read Speed | Space Overhead | Best For |
|----------|------------|------------|----------------|----------|
| STCS | ⚡ Fast | 🐌 Slower | ~2× temp | Write-heavy (logging, events) |
| LCS | Slower | ⚡ Fast | ~10% temp | Read-heavy (profiles, catalog) |
| TWCS | ⚡ Fast | ⚡ Fast* | Minimal | Time-series with TTL (metrics) |

*TWCS reads are fast because data is partitioned by time — you usually only read recent windows.

In [ ]:
# View compaction stats across all tables
result = subprocess.run(
    ["docker", "exec", "cassandra-node1", "nodetool", "compactionstats"],
    capture_output=True, text=True
)
print("Current compaction activity:")
print(result.stdout if result.stdout else "No active compactions (all caught up!)")

## Step 7: Bloom Filters — Avoiding Unnecessary Disk Reads

When reading data, Cassandra needs to check SSTables on disk. But how does it know which
SSTable contains the data it's looking for?

Each SSTable has a **Bloom filter** — a probabilistic data structure that can quickly tell you:
- "This key is **definitely NOT** in this SSTable" (skip it!)
- "This key **might be** in this SSTable" (check it)

Bloom filters have **no false negatives** (never miss data that exists) but can have
**false positives** (occasionally check an SSTable that doesn't have the data).

This is how Cassandra keeps reads fast despite having multiple SSTables.

In [ ]:
# A simple Python demonstration of how Bloom filters work
# (Cassandra uses them internally — this is just to illustrate the concept)

class SimpleBloomFilter:
    """A basic Bloom filter to illustrate the concept."""
    def __init__(self, size=100):
        self.size = size
        self.bits = [False] * size

    def _hashes(self, item):
        """Two hash functions for the item."""
        h1 = hash(item) % self.size
        h2 = hash(f"{item}_salt") % self.size
        return [h1, h2]

    def add(self, item):
        for h in self._hashes(item):
            self.bits[h] = True

    def might_contain(self, item):
        return all(self.bits[h] for h in self._hashes(item))

# Simulate: SSTable has keys 0-99
bf = SimpleBloomFilter(size=1000)
for i in range(100):
    bf.add(f"key_{i}")

# Test: keys we KNOW are in the SSTable
print("Testing keys that ARE in the SSTable:")
for key in ["key_0", "key_50", "key_99"]:
    print(f"  {key}: {'might be here' if bf.might_contain(key) else 'definitely not here'}")

# Test: keys we KNOW are NOT in the SSTable
print("\nTesting keys that are NOT in the SSTable:")
false_positives = 0
for i in range(100, 200):
    if bf.might_contain(f"key_{i}"):
        false_positives += 1

print(f"  Out of 100 absent keys, {false_positives} got false positives")
print(f"  {100 - false_positives} were correctly filtered out (saved {100 - false_positives} disk reads!)")

## 🧠 Key Takeaways

1. **Cassandra writes are append-only.** Updates create new entries; deletes create tombstones. This makes writes very fast.

2. **SSTables are immutable.** Once flushed from the memtable, they're never modified — only merged during compaction.

3. **Tombstones are delete markers.** They stick around for `gc_grace_seconds` to ensure all replicas learn about the delete.

4. **Compaction merges SSTables** to remove old versions and expired tombstones, keeping reads efficient.

5. **Choose your compaction strategy by workload:**
   - **STCS** for write-heavy (logging, events)
   - **LCS** for read-heavy (profiles, lookups)
   - **TWCS** for time-series with TTL (metrics, IoT)

6. **Bloom filters** let Cassandra skip SSTables that definitely don't contain the requested key.

## 🎓 Series Complete!

You've now covered the four pillars of Cassandra:
1. **Data modeling with partition keys** — query-driven design
2. **Wide rows and clustering columns** — sorted data and denormalization
3. **Replication and consistency levels** — CAP theorem trade-offs
4. **Compaction strategies** — LSM trees and storage management

Continue to **Notebook 5** for lightweight transactions, TTL, counters, batches, anti-patterns, and production tuning tips — everything you need before running Cassandra in production.

In [ ]:
cluster.shutdown()
print("Connection closed.")